# Test III: CNN Feature Extractor + VQC

**Key idea:** Replace PCA with a pre-trained CNN to extract 16-dim features.
The CNN learns nonlinear, task-relevant features (vs PCA's linear variance-based ones).

**Pipeline:**
```
Image (1, 150, 150) → CNN (frozen, from Test I) → 16-dim features
    → Angle Encoding (16 qubits) → VQC (3 layers) → ⟨Z⟩ on 3 qubits → 3 classes
```

**Advantage over PCA+VQC:** CNN features are discriminative (trained for this task),
so the VQC starts from a much better feature space.

In [ ]:
import os, time, numpy as np, torch, torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset, Dataset
from sklearn.metrics import roc_curve, auc, roc_auc_score
from sklearn.preprocessing import label_binarize
import matplotlib.pyplot as plt
import cudaq
from cudaq import spin
import warnings; warnings.filterwarnings('ignore')

try: cudaq.set_target('nvidia'); print('GPU target')
except: print('CPU target')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch: {device}')

## 1. Load Pre-trained CNN & Extract Features

In [ ]:
# Import CNN model from Test I
from model_cnn import build_resnet18, ResBlock

# Build CNN and load best weights
cnn = build_resnet18(num_classes=3)
cnn.load_state_dict(torch.load('best_cnn.pt', map_location=device))
cnn = cnn.to(device)
cnn.eval()
print(f'Loaded CNN from best_cnn.pt')

# Create feature extractor: everything before the final Linear layer
# ResNet output before FC: (B, 512) after AdaptiveAvgPool+Flatten
# We add a projection to 16 dims
class CNNFeatureExtractor(nn.Module):
    def __init__(self, cnn_model, out_dim=16):
        super().__init__()
        # All layers except the last Linear
        self.backbone = nn.Sequential(*list(cnn_model.children())[:-1])  # up to Flatten
        self.proj = nn.Linear(512, out_dim)
    
    def forward(self, x):
        with torch.no_grad():
            feat = self.backbone(x)  # (B, 512)
        return torch.sigmoid(self.proj(feat))  # (B, 16) in [0, 1]

# Actually, nn.Sequential doesn't nest cleanly. Let's extract manually.
# The CNN is nn.Sequential: [Conv, BN, ReLU, MaxPool, Res, Res, Res, ..., AvgPool, Flatten, Linear]
# We want everything except the last Linear.
cnn_layers = list(cnn.children())
print(f'CNN has {len(cnn_layers)} top-level layers')
print(f'Last layer: {cnn_layers[-1]}')  # Should be Linear(512, 3)

backbone = nn.Sequential(*cnn_layers[:-1]).to(device)  # all except Linear
backbone.eval()

# Test
with torch.no_grad():
    dummy = torch.randn(2, 1, 150, 150).to(device)
    feat = backbone(dummy)
    print(f'Backbone output: {feat.shape}')  # (2, 512)

In [ ]:
# Train a small projection layer: 512 → 16, sigmoid to [0,1]
# We'll train this briefly with the CNN's own logits as guidance
proj = nn.Sequential(
    nn.Linear(512, 16),
    nn.Sigmoid()  # output in [0, 1] for angle encoding
).to(device)

# Quick train projection to preserve class info
proj_head = nn.Sequential(proj, nn.Linear(16, 3)).to(device)

CLASS_NAMES = ['no', 'sphere', 'vort']
DATA_DIR = 'data'

class LensingDataset(Dataset):
    def __init__(self, root):
        self.samples, self.labels = [], []
        for ci, cn in enumerate(CLASS_NAMES):
            d = os.path.join(root, cn)
            for f in sorted(os.listdir(d)):
                if f.endswith('.npy'):
                    self.samples.append(os.path.join(d, f))
                    self.labels.append(ci)
        self.labels = np.array(self.labels)
    def __len__(self): return len(self.samples)
    def __getitem__(self, i):
        img = np.load(self.samples[i]).astype(np.float32)
        return torch.from_numpy(img), self.labels[i]

train_ds = LensingDataset(os.path.join(DATA_DIR, 'train'))
val_ds = LensingDataset(os.path.join(DATA_DIR, 'val'))
train_img_loader = DataLoader(train_ds, batch_size=128, shuffle=True, num_workers=0)
val_img_loader = DataLoader(val_ds, batch_size=128, shuffle=False, num_workers=0)
print(f'Train: {len(train_ds)}, Val: {len(val_ds)}')

In [ ]:
# Train projection (5 epochs, fast)
opt_proj = torch.optim.Adam(proj_head.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

for ep in range(5):
    proj_head.train()
    c, t = 0, 0
    for imgs, labels in train_img_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        with torch.no_grad():
            feat512 = backbone(imgs)
        out = proj_head(feat512)
        loss = criterion(out, labels)
        opt_proj.zero_grad(); loss.backward(); opt_proj.step()
        c += (out.argmax(1)==labels).sum().item(); t += len(labels)
    print(f'Proj epoch {ep+1}: acc={c/t:.4f}')

proj.eval()
print('Projection trained')

In [ ]:
# Extract 16-dim features for all images
def extract_features(loader):
    all_feat, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device)
            feat512 = backbone(imgs)
            feat16 = proj(feat512)
            all_feat.append(feat16.cpu().numpy())
            all_labels.append(labels.numpy())
    return np.concatenate(all_feat), np.concatenate(all_labels)

print('Extracting CNN features...')
X_train_feat, y_train = extract_features(train_img_loader)
X_val_feat, y_val = extract_features(val_img_loader)
print(f'Train features: {X_train_feat.shape}, range [{X_train_feat.min():.3f}, {X_train_feat.max():.3f}]')
print(f'Val features: {X_val_feat.shape}')

## 2. VQC on CNN Features

In [ ]:
N_QUBITS = 16; N_LAYERS = 3; N_CLASSES = 3
N_PARAMS = N_LAYERS * 2 * N_QUBITS  # 96
BATCH_SIZE = 64; NUM_EPOCHS = 50
LR = 0.1; MOMENTUM = 0.9; EPSILON = 0.02; N_SPSA_AVG = 3
TRAIN_SUBSET = 6000

@cudaq.kernel
def vqc_kernel(data: list[float], params: list[float],
               n_qubits: int, n_layers: int):
    q = cudaq.qvector(n_qubits)
    for layer in range(n_layers):
        base = layer * 2 * n_qubits
        for i in range(n_qubits):
            ry(data[i] * 3.14159265 + params[base + i], q[i])
        for i in range(n_qubits - 1):
            x.ctrl(q[i], q[i + 1])
        x.ctrl(q[n_qubits - 1], q[0])
        for i in range(n_qubits):
            rz(params[base + n_qubits + i], q[i])

CLASS_OBS = [spin.z(i) for i in range(N_CLASSES)]
COMBINED_OBS = sum(CLASS_OBS)

def run_vqc(data_list, params_list):
    r = cudaq.observe(vqc_kernel, COMBINED_OBS, data_list, params_list, N_QUBITS, N_LAYERS)
    return [r.expectation(o) for o in CLASS_OBS]

def forward_batch(X, params_list):
    logits = np.zeros((X.shape[0], N_CLASSES))
    for b in range(X.shape[0]):
        e = run_vqc(X[b].tolist(), params_list)
        for c in range(N_CLASSES): logits[b,c] = e[c]
    return logits

def cross_entropy(logits, labels):
    e = np.exp(logits - logits.max(1, keepdims=True))
    p = e / e.sum(1, keepdims=True)
    return -np.mean(np.log(p[np.arange(len(labels)), labels] + 1e-10))

def spsa_gradient_avg(X_batch, y_batch, params_np, epsilon, n_avg):
    X_np, y_np = X_batch.numpy(), y_batch.numpy()
    grad_sum = np.zeros_like(params_np)
    loss_sum = 0.0
    for _ in range(n_avg):
        delta = np.random.choice([-1.0, 1.0], size=params_np.shape)
        l_p = cross_entropy(forward_batch(X_np, (params_np + epsilon*delta).tolist()), y_np)
        l_m = cross_entropy(forward_batch(X_np, (params_np - epsilon*delta).tolist()), y_np)
        grad_sum += (l_p - l_m) / (2.0 * epsilon) / delta
        loss_sum += (l_p + l_m) / 2.0
    return grad_sum / n_avg, loss_sum / n_avg

def evaluate_vqc(params_np, X, y, bs=256):
    params_list = params_np.tolist()
    all_probs, all_labels = [], []
    for i in range(0, len(X), bs):
        logits = forward_batch(X[i:i+bs].numpy(), params_list)
        e = np.exp(logits - logits.max(1, keepdims=True))
        all_probs.append(e / e.sum(1, keepdims=True))
        all_labels.append(y[i:i+bs].numpy())
    probs = np.concatenate(all_probs)
    labels = np.concatenate(all_labels)
    return (probs.argmax(1)==labels).mean(), probs, labels

print(f'VQC ready: {N_PARAMS} params')

In [ ]:
# Prepare data
idx = []
for c in range(3):
    ci = np.where(y_train == c)[0]
    idx.extend(np.random.choice(ci, TRAIN_SUBSET//3, replace=False).tolist())
np.random.shuffle(idx)

X_tr = torch.tensor(X_train_feat[idx], dtype=torch.float32)
y_tr = torch.tensor(y_train[idx], dtype=torch.long)
X_vl = torch.tensor(X_val_feat, dtype=torch.float32)
y_vl_t = torch.tensor(y_val, dtype=torch.long)
train_loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=BATCH_SIZE, shuffle=True)
print(f'Train: {len(X_tr)}, Val: {len(X_vl)}, Steps/epoch: {len(train_loader)}')

In [ ]:
# Training
params = np.random.randn(N_PARAMS) * 0.1
velocity = np.zeros_like(params)
best_val_acc, best_params = 0.0, params.copy()
history = {'train_loss': [], 'val_acc': []}

ckpt_path = 'cnn_vqc_ckpt.npz'
start_epoch = 0
if os.path.exists(ckpt_path):
    ck = np.load(ckpt_path, allow_pickle=True)
    params, velocity = ck['params'], ck['velocity']
    best_params, best_val_acc = ck['best_params'], float(ck['best_val_acc'])
    start_epoch = int(ck['epoch'])
    history = ck['history'].item()
    print(f'Resumed from epoch {start_epoch}')

for epoch in range(start_epoch, NUM_EPOCHS):
    t0 = time.time()
    eloss, ns = 0., 0
    for Xb, yb in train_loader:
        grad, loss = spsa_gradient_avg(Xb, yb, params, EPSILON, N_SPSA_AVG)
        velocity = MOMENTUM * velocity - LR * grad
        params = params + velocity
        eloss += loss; ns += 1
    
    val_acc, _, _ = evaluate_vqc(params, X_vl[:1500], y_vl_t[:1500])
    if val_acc > best_val_acc:
        best_val_acc = val_acc; best_params = params.copy()
    history['train_loss'].append(eloss/ns)
    history['val_acc'].append(val_acc)
    np.savez(ckpt_path, params=params, velocity=velocity,
             best_params=best_params, best_val_acc=best_val_acc,
             epoch=epoch+1, history=history)
    print(f'Epoch {epoch+1:02d}/{NUM_EPOCHS} ({time.time()-t0:.0f}s) | '
          f'Loss: {eloss/ns:.4f} | Val Acc: {val_acc:.4f} | Best: {best_val_acc:.4f}')

print(f'\nBest Val Accuracy: {best_val_acc:.4f}')

In [ ]:
# Training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history['train_loss'], 'b-'); ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.set_title('Training Loss'); ax1.grid(alpha=0.3)
ax2.plot(history['val_acc'], 'r-'); ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy')
ax2.set_title(f'Val Accuracy (Best: {best_val_acc:.4f})')
ax2.axhline(y=1/3, color='gray', ls='--', alpha=0.5, label='Random'); ax2.legend(); ax2.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 3. Final Evaluation — ROC Curve & AUC

In [ ]:
print('Evaluating best model on full val set...')
final_acc, all_probs, all_labels = evaluate_vqc(best_params, X_vl, y_vl_t)
all_labels_bin = label_binarize(all_labels, classes=[0,1,2])

fig, ax = plt.subplots(figsize=(8, 6))
colors = ['#e41a1c', '#377eb8', '#4daf4a']
for i, cn in enumerate(CLASS_NAMES):
    fpr, tpr, _ = roc_curve(all_labels_bin[:,i], all_probs[:,i])
    ax.plot(fpr, tpr, color=colors[i], lw=2, label=f'{cn} (AUC={auc(fpr,tpr):.4f})')
macro_auc = roc_auc_score(all_labels_bin, all_probs, multi_class='ovr', average='macro')
ax.plot([0,1],[0,1],'k--',lw=1)
ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
ax.set_title(f'ROC — CNN+VQC (Acc={final_acc:.4f}, Macro AUC={macro_auc:.4f})')
ax.legend(loc='lower right'); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig('roc_cnn_vqc.png', dpi=150); plt.show()

for i, cn in enumerate(CLASS_NAMES):
    fpr, tpr, _ = roc_curve(all_labels_bin[:,i], all_probs[:,i])
    print(f'  {cn} AUC: {auc(fpr,tpr):.4f}')
print(f'Macro AUC: {macro_auc:.4f}')

## 4. Discussion

### CNN+VQC vs PCA+VQC

| Aspect | PCA+VQC | CNN+VQC |
|--------|---------|--------|
| Feature extraction | Linear (PCA) | Nonlinear (pre-trained CNN) |
| Feature quality | Variance-based, task-agnostic | Discriminative, task-specific |
| Information preserved | ~85% variance | Class-relevant features only |
| VQC's job | Learn full classification boundary | Fine-tune from good features |

The CNN has already learned to separate the 3 classes (91.67% accuracy).
Its 512-dim features are projected to 16 dims while preserving class structure.
The VQC then operates on these discriminative features, which should be much
easier to classify than raw PCA components.